# Project 10 — Step-by-Step Walkthrough

**Dynamic Trend & Event Detector**  
This notebook explains **the order of work**, **what each notebook/script does**, and **why we do each step** — use it for viva, demos, and onboarding.

---

## How this notebook is meant to be used

- Read top-to-bottom once to understand the **full story**.
- When presenting, jump to the step that matches the notebook you are showing.
- The last code cell optionally **checks** that key outputs exist (no model training here).

## Big picture: what we are trying to do

| Course goal | How this project addresses it |
|---|---|
| Detect **evolving topics** over time | LDA topics + SBERT clusters + **semantic velocity** (weekly shift) + **growth velocity** (cluster size) |
| **Baseline** frequency methods | TF-IDF global ranking + **burst** days |
| **Advanced** probabilistic topics | LDA with **Gensim C_V** coherence |
| **Deep learning** embeddings | SBERT sentence vectors → UMAP → K-Means |
| **Hybrid / edge** | Pipeline: lexical + probabilistic + neural + **GDELT** external news + impact scoring |
| **Extra mile** | **Event impact** $S_I$ = SBERT uniqueness × \|GDELT tone\| |

**Why a pipeline instead of one “super model”?**  
Each method has different strengths: TF-IDF is fast and interpretable; LDA gives document–topic mixtures; SBERT captures **meaning**; GDELT adds **global** context. Combining stages is clearer for analysis and grading than a single black box.

---
## Step 0 — `00_run_all.ipynb` (optional orchestrator)

**What:** Runs other notebooks or scripts in sequence so you do not forget a step.

**Why we do this:** Reproducibility. A reviewer (or you, next week) can replay the full experiment from one place. Same idea as `python run_all.py` for the `src/` scripts.

---
## Step 1 — `01_eda.ipynb` · `src/eda.py`

**What:** Load the headline CSV, parse dates, plot **how many headlines per day** and **headline length** distribution; peek at raw rows.

**Why we do this:**
- **Before** fitting any model, we must know if the data is balanced over time (missing years? spikes?).
- Length distribution catches encoding issues or empty text.
- EDA does **not** train anything — it prevents garbage-in-garbage-out and justifies modelling choices (e.g. full 2003–2021 span).

---
## Step 2 — `02_baseline_tfidf.ipynb` · `src/baseline.py`

**What:** Clean text → **TF-IDF** matrix → global **term ranking** → **burst** analysis (when do top terms spike per day?).

**Why we do this:**
- Course requires a **simple frequency-based** baseline. TF-IDF is stronger than raw counts because **IDF** down-weights words that appear everywhere (“says”, “new”).
- **Bursts** approximate “breaking” or heavy-coverage days using only lexical signal — cheap and interpretable.
- This sets a **floor**: later models should add value *beyond* what TF-IDF already sees.

---
## Step 3 — `03_advanced_ml_lda.ipynb` · `src/advanced_ml.py`

**What:** **Count** vectors (not TF-IDF) → **Latent Dirichlet Allocation** → topics as word distributions → **coherence** (Gensim **C_V**, fallback UMass) → discrimination / confidence plots.

**Why we do this:**
- Course asks for **probabilistic topic models**. LDA treats each headline as a **mixture of topics** — realistic for news.
- Coherence scores tell us if topics are **human-interpretable**, not just low perplexity.
- **Why CountVectorizer, not TF-IDF for LDA?** LDA’s math assumes **multinomial** word counts; TF-IDF breaks that assumption.

**Note:** Gensim may need **Python 3.11** kernel (`dtdetector311`) if your default Python cannot install Gensim wheels.

---
## Step 4 — `08_deep_learning_kmeans.ipynb` · `src/deep_learning.py`

**What:** **Stratified sample** (~50k headlines, equal per year) → **SBERT** sentence embeddings (384-d) → **UMAP** (2D/3D for plots) → **K-Means** (search K, e.g. 2–15) → **TF-IDF labels per cluster** (names only) → **growth velocity** & **semantic velocity** → save `semantic_velocity.csv`.

**Why we do this:**
- Course **DL** requirement: **embedding-based clustering**. SBERT captures **semantics** (“troops” vs “soldiers”) that LDA/TD-IDF treat as unrelated tokens.
- **Why sample 50k, not 1.24M?** Encoding full corpus with SBERT is hours of GPU/CPU time; **stratified** sampling keeps **every year** represented — fair for long-term trends.
- **Semantic velocity** $V_s = 1 - \cos(\bar{e}_{t-1}, \bar{e}_t)$ measures **narrative rupture** — when the *average meaning* of the week jumps. That is the **temporal evolution** part of the hybrid story.
- **Why TF-IDF for cluster “Topic A/B” labels?** Clusters live in embedding space; TF-IDF simply picks **readable words** for slides — the **clustering** is still SBERT-based.

---
## Step 5 — `04_gdelt_processor.ipynb` · `src/gdelt_processor.py` + `src/gdelt_fetcher.py`

**What:** Download **live** GDELT GKG (or fall back to local `.gkg.csv`) → parse themes + tone → `data/gdelt_processed.csv`.

**Why we do this:**
- The ABC corpus is **historical**; GDELT is **global and near real-time** (updates ~every 15 minutes). That satisfies the **journalism / policy / external validation** angle.
- Themes and tone are **structured** — comparable across languages and sources — unlike scraping random HTML.

**Why not before Step 4?** You *can* fetch GDELT anytime; in **our** design, deep learning runs first so **notebook 09** can use **rupture weeks** from `semantic_velocity.csv` together with GDELT for **verification** and impact scoring.

---
## Step 6 — `05_gdelt_analysis.ipynb` · `src/gdelt_analysis.py`

**What:** Aggregate **theme counts**, plot top themes, average **tone** per theme.

**Why we do this:** Raw GDELT rows are noisy; analysts need **summaries** (which themes dominate this snapshot? how negative/positive?). These plots go into reports and presentations.

---
## Step 7 — `09_gdelt_verification_impact.ipynb` · `src/event_impact_scoring.py`

**What:** Read top **rupture weeks** from deep learning → optionally show ABC headlines in those windows → encode GDELT theme strings with **SBERT** → **impact score** $S_I = \text{uniqueness} \times |\text{tone}|$.

**Why we do this:**
- **Verification:** Links a **math spike** (semantic velocity) to **human-readable** headlines — proves the metric is not arbitrary.
- **Impact scoring (extra mile):** Ranks **which GDELT events** are both **semantically unusual** in the snapshot and **emotionally strong** (tone magnitude).
- **Why SBERT for scoring themes?** GDELT themes are short strings; SBERT knows “health” and “pandemic” are related better than TF-IDF token overlap.

---
## Step 8 — `06_event_impact_scoring.ipynb` (optional baseline)

**What:** Same impact idea but **TF-IDF** vectors for uniqueness instead of SBERT.

**Why we do this:** Shows **progression**: a fast baseline vs the **production** SBERT method in notebook 09 — good for reports and ablation-style discussion.

---
## Step 9 — `07_visualize_results.ipynb`

**What:** Dashboard: load figures/CSVs from `reports/` — baseline, LDA, deep learning, GDELT, impact.

**Why we do this:** One place for a **demo or examiner** to see the whole project without opening every notebook.

---
## Snapshot — notebooks **11** & **12** (topic time series → events)

**What we built (one chain)**

| Step | Notebook | Main outputs |
|---|---|---|
| **11** | `11_topic_modeling_lda_bertopic.ipynb` | `data/df_clean.pkl`, SBERT cluster bundle under `models/`, **`reports/topic_modeling/11_lda_sbert/topics_over_time.csv`**, LDA vs SBERT comparison CSVs |
| **12** | `12_phase4_trend_and_events.ipynb` | **`spike_events.csv`** (per-topic z on growth-velocity > 2.5), optional **`anchor_ground_truth_detection.csv`** |

**What is *not* redundant**

- **`spike_events.csv`**: built only in notebook **12** — per-topic z-score on growth-velocity > 2.5 (after notebook 11 writes `topics_over_time.csv`).
- **Anchor CSV**: human date windows + **best matching row** from pipeline tables per window — for evaluation slides, not a fourth spike detector.

**Ignore for reading logic**: auto-generated `.ipynb_checkpoints/` (gitignored).


---
## Supporting pieces (not sequential pipeline steps)

| Asset | What | Why |
|---|---|---|
| `refresh_gdelt.sh` | Clears cache, fetches fresh GDELT, re-runs downstream | Update global layer **without** retraining LDA/SBERT |
| `generate_all_visuals.py` | Regenerate plots from cached CSVs | Fast figure refresh after small report tweaks |
| `visualize_results.py` | Legacy helper plots | Optional extra PNGs if CSVs exist |
| `docs/PROJECT_WALKTHROUGH.md` | Same story as this notebook in Markdown | For readers who prefer GitHub/docs over Jupyter |

---

## “Hybrid” in one paragraph

We do **not** use the old monolithic `hybrid_temporal.py`. Instead, **hybrid** means: **TF-IDF** (lexical) + **LDA** (probabilistic topics) + **SBERT** (neural semantics) + **time** (semantic/growth velocity) + **GDELT** (external world). Each piece is testable; together they match the course’s **Social Media Analytics / Journalism / Policy** story.

---
## Optional: verify that key outputs exist (no training)

Run the next cell from the **project root** (or the cell will `chdir` to parent if you launched inside `notebooks/`).

In [ ]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working directory:', os.getcwd())

checks = [
    ('Full headline corpus', 'data/abcnews-date-text 2.csv'),
    ('GDELT processed', 'data/gdelt_processed.csv'),
    ('Semantic velocity (run notebook 08 first)', 'reports/deep_learning/semantic_velocity.csv'),
    ('Event impact scores (run notebook 09)', 'reports/event_impact_scores.csv'),
    ('Topic time series (notebook 11)', 'reports/topic_modeling/11_lda_sbert/topics_over_time.csv'),
    ('Spikes + anchors (notebook 12)', 'reports/topic_modeling/12_spikes_anchors/spike_events.csv'),
]

for label, path in checks:
    ok = os.path.isfile(path)
    status = 'OK' if ok else 'MISSING — run the step above'
    print(f'  [{status:32}] {label}: {path}')